In [ ]:
import json, re, random, os
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Any
import statistics

#Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

BASE = Path("/Users/matteogevi/Aurora-History-MVP/data/out")
CHUNKS_PATH = BASE / "chunks.paragraphs.jsonl"
TOC_PATH = BASE / "toc.json"

In [ ]:
'''Chunking Display'''

rows = [json.loads(l) for l in open(CHUNKS_PATH, "r", encoding="utf-8") if l.strip()]
print(f"✅ Loaded {len(rows)} chunks")

df = pd.DataFrame(rows)
print("Columns:", list(df.columns))

# -------- quick sanity summary --------
print(
    f"rows: {len(df)} | "
    f"unique sections: {df['section_id'].nunique()} | "
    f"avg chars: {df['text'].str.len().mean():.1f} | "
    f"median: {df['text'].str.len().median():.0f} | "
    f">2000 chars: {(df['text'].str.len()>2000).sum()} | "
    f"empty: {(df['text'].str.len()==0).sum()}"
)

# -------- sample preview --------
def preview_row(r):
    t = (r.get("text") or "").replace("\n", " ")
    return {
        "chunk_id": r["chunk_id"],
        "section_id": r["section_id"],
        "section_title": r.get("section_title"),
        "level": r.get("level"),
        "page_range": r.get("page_range"),
        "len": len(t),
        "text_preview": t[:200],
    }

preview_df = pd.DataFrame([preview_row(r) for r in rows])
preview_df.head(10)

In [ ]:
'''Chunk length & duplicate sniff'''

rows = [json.loads(l) for l in open(CHUNKS_PATH, "r", encoding="utf-8") if l.strip()]
print("rows loaded:", len(rows))

# ---- Auto-detect TEXT_KEY (favor 'text' if present) ----
lens = defaultdict(list)
for r in rows:
    for k, v in r.items():
        if isinstance(v, str):
            lens[k].append(len(v.strip()))

def score(arr):
    nonempty = sum(1 for x in arr if x > 0)
    avg = statistics.mean(arr) if arr else 0.0
    return (nonempty, avg)

candidates = {k: score(arr) for k, arr in lens.items()}

if "text" in candidates:
    TEXT_KEY = "text"  # your new schema; be explicit if available
else:
    TEXT_KEY = max(candidates, key=lambda k: (candidates[k][0], candidates[k][1])) if candidates else None

print("Using TEXT_KEY:", TEXT_KEY, "->", candidates.get(TEXT_KEY))

# ---- Chunk length & duplicate sniff ----
def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

if TEXT_KEY:
    lengths = [len((r.get(TEXT_KEY) or "").strip()) for r in rows]
    print(
        "n:", len(rows),
        "| avg chars:", round(statistics.mean(lengths), 1) if lengths else 0,
        "| median:", int(statistics.median(lengths)) if lengths else 0,
        "| >2000 chars:", sum(int(L > 2000) for L in lengths),
        "| empty:", sum(int(L == 0) for L in lengths),
    )

    seen = set()
    dups = 0
    for r in rows:
        key = normalize(r.get(TEXT_KEY) or "")[:400]  # prefix to keep memory low
        if key in seen:
            dups += 1
        seen.add(key)
    print("potential duplicates:", dups)
else:
    print("No textual key detected; rows may be malformed.")

# ---- Nice DataFrame preview for your schema ----
def preview_row(r):
    txt = (r.get(TEXT_KEY) or "").replace("\n", " ")
    return {
        "chunk_id": r.get("chunk_id"),
        "section_id": r.get("section_id") or r.get("section_node_id"),
        "section_title": r.get("section_title"),
        "level": r.get("level"),
        "page_range": r.get("page_range"),
        "len": len(txt),
        "text_preview": txt[:220],
    }

df = pd.DataFrame([preview_row(r) for r in rows])
display(df.head(12))

# ---- Quick per-section summary (helps spot skew) ----
sec_summary = (
    df.groupby(["level", "section_id", "section_title"], dropna=False)["len"]
      .agg(n_chunks="count", avg_len="mean", median_len="median")
      .reset_index()
      .sort_values(["level", "n_chunks"], ascending=[True, False])
)
display(sec_summary.head(15))


In [ ]:
'''Embedding norms + self-nearest-neighbor sanity'''

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [r.get("content","") for r in rows[:500]]  # subset for speed
X = model.encode(texts, normalize_embeddings=True).astype("float32")

# Check unit norms
norms = np.linalg.norm(X, axis=1)
print("norm mean:", float(norms.mean()), "min:", float(norms.min()), "max:", float(norms.max()))

# Self NN (exclude self)
S = X @ X.T
np.fill_diagonal(S, -1.0)
top_idx = S.argmax(axis=1)
sample = [(i, int(top_idx[i]), float(S[i, top_idx[i]])) for i in range(min(10, len(texts)))]
print("top neighbors (i -> j, cos):", sample[:5])

In [ ]:
'''“Same section” vs random similarity'''

sec = [r.get("section_node_id") for r in rows[:500]]
pairs_same, sims_same = 0, []
pairs_diff, sims_diff = 0, []

for _ in range(100):
    i = random.randrange(len(sec)); j = random.randrange(len(sec))
    if i == j: continue
    sim = float(X[i] @ X[j])
    if sec[i] and sec[i] == sec[j]:
        pairs_same += 1; sims_same.append(sim)
    else:
        pairs_diff += 1; sims_diff.append(sim)

def m(a): return round(sum(a)/len(a), 4) if a else None
print("same-sec avg cos:", m(sims_same), "n:", pairs_same)
print("diff-sec avg cos:", m(sims_diff), "n:", pairs_diff)